<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module12/Lab7.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 7 — QAOA: Ideal Simulator, Noise, and Real Hardware (Instructor)
**Quantum Optimization and Simulation — QAOA Laboratory Series**

Instructor version with answer key.

**Format:** 10–15 minute instructor walkthrough + about 45–60 minutes of independent work.

**Notebook style:** Most code is supplied. Cells marked **YOUR TURN** contain a small value, line, or function for you to complete.

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Compare an ideal Aer simulation with a noisy simulation.
- Transpile the same QAOA circuit for an IBM backend.
- Optionally submit a real-hardware job.
- Relate circuit depth/two-qubit gates to NISQ performance.

**Important:** The real-hardware section is optional if you do not have IBM Quantum access or a queue is long.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

edges5 = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

def build_qaoa_5(gamma=0.68, beta=0.39):
    qc = QuantumCircuit(5)
    qc.h(range(5))
    for i, j in edges5:
        qc.cx(i, j)
        qc.rz(-gamma, j)
        qc.cx(i, j)
    for q in range(5):
        qc.rx(2 * beta, q)
    qc.measure_all()
    return qc

qc = build_qaoa_5()

## Part A — Ideal Aer

In [ ]:
ideal_backend = AerSimulator()
tqc = transpile(qc, ideal_backend, optimization_level=1)
ideal_counts = ideal_backend.run(tqc, shots=4096, seed_simulator=SEED).result().get_counts()
plot_histogram(ideal_counts)

## Part B — A simple noisy Aer model

In [ ]:
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.002, 1), ["h", "rz", "rx"])
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ["cx"])

noisy_backend = AerSimulator(noise_model=noise_model)
tqc_noisy = transpile(qc, noisy_backend, optimization_level=1)
noisy_counts = noisy_backend.run(tqc_noisy, shots=4096, seed_simulator=SEED).result().get_counts()

plot_histogram([ideal_counts, noisy_counts], legend=["Ideal", "Noisy"])

**Expected:** the noisy distribution is typically flatter/spread out compared with the ideal one.

### YOUR TURN
Increase the two-qubit depolarizing error from `0.02` to `0.05`. What changes?

## Part C — Optional IBM hardware

In [ ]:
# OPTIONAL REAL-HARDWARE SECTION
#
# from qiskit_ibm_runtime import QiskitRuntimeService
# from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler
# from qiskit.transpiler import generate_preset_pass_manager
#
# service = QiskitRuntimeService()
# backend = service.least_busy(min_num_qubits=5, operational=True, simulator=False)
# print("Using:", backend.name)
#
# pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
# isa_qc = pm.run(qc)
#
# print("Original depth:", qc.depth())
# print("Transpiled depth:", isa_qc.depth())
# print("Transpiled ops:", isa_qc.count_ops())
#
# runtime_sampler = RuntimeSampler(mode=backend)
# job = runtime_sampler.run([isa_qc], shots=4096)
# hardware_result = job.result()
# hardware_counts = hardware_result[0].data.meas.get_counts()
# plot_histogram([ideal_counts, hardware_counts], legend=["Ideal", "Hardware"])

## Questions
1. Why are two-qubit gate errors particularly important for this QAOA circuit?
2. Why can transpilation increase circuit depth?
3. Why should an Aer noise model be viewed as an approximation rather than a perfect prediction of hardware?

## Instructor solutions

1. The cost layer contains many CNOTs; two-qubit gates are typically noisier than single-qubit gates, so their errors accumulate.
2. Hardware has limited connectivity and a restricted native gate set. The transpiler may insert routing/SWAP operations and decompose gates.
3. A simulator noise model includes only selected error mechanisms and usually cannot reproduce every drift, crosstalk, calibration change, leakage, or correlated error on a live device.